In [ ]:
from urllib.parse import urljoin
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import re
import time
import pandas as pd
import requests


In [ ]:

# ============================================================
# GLOBAL SOURCE SETTINGS
# ============================================================

SOURCE_NAME = "SSC"

# Human-facing official page. Stored as source evidence in the catalogue.
PARENT_URL = "https://ssc.gov.in/home/notice-board"

# Machine-readable source used by this SSC connector.
API_ENDPOINT = "https://ssc.gov.in/api/general-website/portal/records"

# Choose ONE:
PAGE_LIMIT = "max"  # Automatically fetch until no more notices exist.
# PAGE_LIMIT = 72        # Or fetch a fixed number of pages.
# PAGE_LIMIT = 5         # Useful while testing.

RECORDS_PER_PAGE = 10

# Protection against an accidental endless loop when PAGE_LIMIT = "max".
SAFETY_MAX_PAGES = 1_000

REQUEST_DELAY_SECONDS = 0.5
TIMEOUT_SECONDS = 30

ATTACHMENT_BASE_URL = "https://ssc.gov.in/api/attachment/"

NOTICE_CATALOGUE_FILE = Path("data/ssc_notice_catalogue.csv")
PDF_CATALOGUE_FILE = Path("data/ssc_pdf_catalogue.csv")

HEADERS = {
    "User-Agent": "GovJobIntelligence/0.1 (educational project)",
    "Accept": "application/json,text/plain,*/*",
}

# ============================================================
# VALIDATE PAGE_LIMIT
# ============================================================

if PAGE_LIMIT != "max":
    if not isinstance(PAGE_LIMIT, int) or PAGE_LIMIT < 1:
        raise ValueError(
            "PAGE_LIMIT must be 'max' or a positive integer, such as 72."
        )


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def build_api_params(page_number):
    """
    SSC-specific API parameters.

    For a different organisation, change this function if its API expects
    different parameter names or does not expose JSON.
    """
    return {
        "page": page_number,
        "limit": RECORDS_PER_PAGE,
        "contentType": "notice-boards",
        "key": "createdAt",
        "order": "DESC",
        "pageType": "filter",
        "isAttachment": "true",
        "attributes": (
            "id,headline,examId,contentType,redirectUrl,"
            "startDate,endDate,language,createdAt"
        ),
        "queryKey": "startDate,endDate",
        "customKey": "createdAt",
        "exams": "false",
        "date": "false",
        "language": "english",
    }


def get_notice_page(page_number, session):
    """Fetch one page of SSC Notice Board data."""
    response = session.get(
        API_ENDPOINT,
        params=build_api_params(page_number),
        timeout=TIMEOUT_SECONDS
    )
    response.raise_for_status()

    payload = response.json()

    if payload.get("statusCode") != "200":
        raise RuntimeError(
            f"{SOURCE_NAME} API error on page {page_number}: {payload}"
        )

    return payload.get("data", [])


def classify_headline(headline):
    """
    Conservative first-pass classification using only the notice title.
    PDF extraction will validate or improve this later.
    """
    text = (headline or "").lower()

    if "final result" in text:
        return "final_result"

    if "declaration of result" in text or "result" in text:
        return "result"

    if "answer key" in text or "response sheet" in text:
        return "answer_key"

    if "admission certificate" in text or "admit card" in text:
        return "admit_card_or_exam_city"

    if (
            "exam date" in text
            or "revised dates" in text
            or "examination schedule" in text
            or "schedule of examination" in text
    ):
        return "exam_schedule"

    if "vacancy" in text:
        return "vacancy_update"

    if "corrigendum" in text or "amendment" in text:
        return "corrigendum_or_amendment"

    if "document verification" in text:
        return "document_verification_notice"

    if "shortlist" in text or "shortlisted" in text:
        return "shortlisted_for_next_stage"

    if "syllabus" in text or "scheme of examination" in text:
        return "syllabus_or_exam_scheme"

    return "other_or_needs_review"


def extract_exam_family(headline):
    """First-pass exam-family extraction from the notice title."""
    text = (headline or "").lower()

    exam_patterns = {
        "CGL": ["combined graduate level", "cgl"],
        "CHSL": ["combined higher secondary", "chsl"],
        "MTS": ["multi-tasking staff", "mts"],
        "JE": ["junior engineer", "je examination"],
        "CPO": ["sub-inspector in delhi police", "central armed police forces"],
        "GD": ["constable (gd)", "rifleman (gd)"],
        "STENOGRAPHER": ["stenographer"],
        "SELECTION_POST": ["selection post"],
        "JHT": ["junior hindi translator", "jht"],
        "CAPF": ["capf"],
        "DEPARTMENTAL": ["departmental competitive examination"],
    }

    matches = [
        exam_name
        for exam_name, patterns in exam_patterns.items()
        if any(pattern in text for pattern in patterns)
    ]

    return ", ".join(matches) if matches else "unknown"


def extract_years(headline):
    """
    Keep every year mentioned in the title.
    Do not assume the first year is the recruitment year.
    """
    years = re.findall(r"\b20\d{2}\b", headline or "")
    return ", ".join(sorted(set(years))) if years else ""


# ============================================================
# FETCH ALL PAGES
# ============================================================

session = requests.Session()
session.headers.update(HEADERS)

all_notices = []
failed_pages = []
page_number = 1

while True:
    # Stop at the user's chosen fixed page limit.
    if isinstance(PAGE_LIMIT, int) and page_number > PAGE_LIMIT:
        print(f"Reached your PAGE_LIMIT of {PAGE_LIMIT}. Finished.")
        break

    # Safety guard when PAGE_LIMIT = "max".
    if page_number > SAFETY_MAX_PAGES:
        raise RuntimeError(
            f"Stopped at SAFETY_MAX_PAGES={SAFETY_MAX_PAGES}. "
            "Check whether the source API is repeating pages."
        )

    try:
        page_records = get_notice_page(page_number, session)

        # This makes PAGE_LIMIT = "max" automatic.
        if not page_records:
            print(f"Page {page_number}: no more records. Finished.")
            break

        all_notices.extend(page_records)

        print(
            f"Page {page_number}: "
            f"{len(page_records)} notices fetched "
            f"| total so far: {len(all_notices)}"
        )

        page_number += 1
        time.sleep(REQUEST_DELAY_SECONDS)

    except Exception as error:
        failed_pages.append({
            "page_number": page_number,
            "error": str(error)
        })

        print(f"Page {page_number}: FAILED | {error}")
        print("Stopped so we do not silently create an incomplete catalogue.")
        break

if not all_notices:
    raise RuntimeError("No notices were retrieved.")

# ============================================================
# BUILD FULL NOTICE CATALOGUE
# ============================================================

notice_rows = []

for notice in all_notices:
    headline = (notice.get("headline") or "").strip()

    notice_rows.append({
        "source": SOURCE_NAME,
        "notice_id": notice.get("id"),
        "headline": headline,
        "exam_id_from_source": notice.get("examId"),
        "exam_family_rule_based": extract_exam_family(headline),
        "mentioned_years": extract_years(headline),
        "document_type_rule_based": classify_headline(headline),
        "classification_source": "headline_rules_v1",
        "classification_status": "needs_pdf_validation",
        "content_type": notice.get("contentType"),
        "language": notice.get("language"),
        "notice_created_at": notice.get("createdAt"),
        "start_date_from_api": notice.get("startDate"),
        "end_date_from_api": notice.get("endDate"),
        "redirect_url": notice.get("redirectUrl"),
        "attachment_count": len(notice.get("attachments", [])),
        "parent_url": PARENT_URL,
        "source_api": API_ENDPOINT,
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat()
    })

notices_df = (
    pd.DataFrame(notice_rows)
    .drop_duplicates(subset=["notice_id"])
    .sort_values("notice_created_at", ascending=False)
)

# ============================================================
# BUILD PDF CATALOGUE
# ============================================================

pdf_rows = []

for notice in all_notices:
    headline = (notice.get("headline") or "").strip()

    for attachment in notice.get("attachments", []):
        if (attachment.get("type") or "").lower() != "application/pdf":
            continue

        relative_path = (
                attachment.get("path") or ""
        ).replace("\\", "/").lstrip("/")

        pdf_rows.append({
            "source": SOURCE_NAME,
            "notice_id": notice.get("id"),
            "headline": headline,
            "exam_family_rule_based": extract_exam_family(headline),
            "mentioned_years": extract_years(headline),
            "document_type_rule_based": classify_headline(headline),
            "classification_status": "needs_pdf_validation",
            "notice_created_at": notice.get("createdAt"),
            "attachment_id": attachment.get("id"),
            "original_filename": attachment.get("fileName"),
            "attachment_type": attachment.get("type"),
            "attachment_size_bytes": attachment.get("size"),
            "attachment_path_from_api": attachment.get("path"),
            "pdf_url": urljoin(ATTACHMENT_BASE_URL, relative_path),
            "local_pdf_path": "",
            "download_status": "not_downloaded",
            "parent_url": PARENT_URL,
            "source_api": API_ENDPOINT,
            "retrieved_at_utc": datetime.now(timezone.utc).isoformat()
        })

pdf_catalogue_df = (
    pd.DataFrame(pdf_rows)
    .drop_duplicates(subset=["attachment_id"])
    .sort_values("notice_created_at", ascending=False)
)

# ============================================================
# SAVE RESULTS
# ============================================================

NOTICE_CATALOGUE_FILE.parent.mkdir(parents=True, exist_ok=True)

notices_df.to_csv(NOTICE_CATALOGUE_FILE, index=False, encoding="utf-8")
pdf_catalogue_df.to_csv(PDF_CATALOGUE_FILE, index=False, encoding="utf-8")

# ============================================================
# DISPLAY SUMMARY
# ============================================================

print("\n" + "=" * 65)
print(f"{SOURCE_NAME} CATALOGUE COMPLETE")
print("=" * 65)
print(f"Notices fetched: {len(notices_df)}")
print(f"PDF attachments: {len(pdf_catalogue_df)}")
print(f"Pages processed: {page_number - 1}")
print(f"Failed pages: {len(failed_pages)}")
print(f"\nNotice catalogue: {NOTICE_CATALOGUE_FILE.resolve()}")
print(f"PDF catalogue:    {PDF_CATALOGUE_FILE.resolve()}")

if failed_pages:
    print("\nFailed pages:")
    display(pd.DataFrame(failed_pages))

print("\nDocument-type distribution:")
display(
    notices_df["document_type_rule_based"]
    .value_counts()
    .rename_axis("document_type")
    .reset_index(name="notice_count")
)

print("\nExam-family distribution:")
display(
    notices_df["exam_family_rule_based"]
    .value_counts()
    .rename_axis("exam_family")
    .reset_index(name="notice_count")
)

print("\nLatest notices:")
display(
    notices_df[
        [
            "notice_created_at",
            "headline",
            "exam_family_rule_based",
            "document_type_rule_based",
            "attachment_count"
        ]
    ].head(20)
)